# Stage C — A100 capacity and horizon matrix
Runs horizons 2/3/4 on identical hardware and data with checkpoint timing and FP32-memory-state evidence. The unaccelerated reference recurrence is a correctness oracle, not a full-geometry capacity workload; Notebook 01 and focused tests cover it. Each exact-SDPA probe runs in its own logged process and appends its completed evidence.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='cc4ba30c153b0d029124c72292d364ca0963064c'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
RUN_NAME='c5_a100_capacity'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess,sys
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'remote','set-url','origin',REPO_URL],check=True)
subprocess.run(['git','-C',str(repo),'fetch','--all'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
selection_path=Path(DRIVE_ROOT)/'runs'/'c1_tokenizers_cpu'/'tokenizer_selection.json'
selection=json.loads(selection_path.read_text(encoding='utf-8'))
selected=selection.get('selected_tokenizer')
if not isinstance(selected,str) or not selected: raise ValueError('tokenizer_selection.json has no selected_tokenizer')
dataset=Path(DRIVE_ROOT)/'stage_c_dataset'/'ordered_streams'/selected
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError(f'Run Notebook 00b first; missing {dataset}/token_stream_manifest.json')
DATASET_DIR=str(dataset)
print('Resolved Handoff 00b dataset:',DATASET_DIR)
output=f'{DRIVE_ROOT}/runs/{RUN_NAME}'
Path(output).mkdir(parents=True,exist_ok=True)
def logged(label,command):
    subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',output,'--label',label,'--repo',str(repo),'--',*command],check=True)
logged('hardware_preflight',['seqtrainer-titans-stage-c-hardware-preflight','--require','A100','--output',f'{output}/hardware.json'])

In [ ]:
# Keep each full-geometry probe below Colab's long-child-process watchdog and preserve prior results.
# reference_fp32 is deliberately excluded: its unaccelerated functional recurrence was killed after 25 minutes before one step on this A100.
for horizon in ('2','3','4'):
    for variant in ('exact_sdpa_fp32','exact_sdpa_bfloat16'):
        logged(f'a100_{variant}_h{horizon}',['seqtrainer-titans-stage-c-capacity','--dataset-dir',DATASET_DIR,'--output-dir',output,'--require','A100','--horizons',horizon,'--variants',variant,'--steps','2','--batch-size','1','--append'])
print('SHARE THIS DIRECTORY:',output)